# RoBERTa Finetuning and testing
made with help from:
- https://github.com/cltl/ma-ml4nlp-labs/blob/main/code/assignment3/other_systems/bert_finetunen.ipynb
- https://github.com/julianschelb/roberta-ner-multilingual/tree/main

In [19]:
# ! pip install scikit-learn
import torch
from transformers import pipeline
from datasets import load_dataset
from evaluate import load
import pandas as pd
import sklearn
from datasets import Dataset
from transformers import RobertaTokenizerFast
import numpy as np
from transformers import AutoModelForTokenClassification
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

In [41]:
task = "ner"
model_checkpoint ="FacebookAI/roberta-base"
batch_size = 16

In [ ]:
train_file = r"data\fixed_combined_training_set.conll"
val_file =r"data\fixed_combined_validation_set.conll"

In [4]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels

In [ ]:
train_tokens, train_labels = get_list_of_sentences(train_file)
val_tokens, val_labels = get_list_of_sentences(val_file)

train_dataset = Dataset.from_dict({"tokens": train_tokens, "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": val_tokens, "ner_tags": val_labels})

In [45]:
label_list = sorted(set(label for sentence in train_labels for label in sentence))
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

In [46]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base", add_prefix_space=True)

In [ ]:
def tokenize_and_align_labels(examples):
    """
    this tokenizes the sentences and aligns the labels with the tokenized subwords
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    #tokenized the lists of sentences
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]): #for every label list in the ner_tags we have
        word_ids = tokenized_inputs.word_ids(batch_index=i) #mapping every subword to the original word index
        aligned_labels = [] 
        previous_word_id = None #looking at the previous word id to detect new ones
        for word_id in word_ids: #going through each token
            #special tokens have a None word id, they are set to -100 to be ignored
            #in the loss function
            if word_id is None: #special tokens
                aligned_labels.append(-100)
            elif word_id != previous_word_id: #if not equal to same as prev, new word!
                aligned_labels.append(label_to_id[word_labels[word_id]]) #assign it the gold lab of the first subword
            else: #continued subword of the same word
                label = word_labels[word_id] #getting the OG label
                if label.startswith("B-"): #check if the label begins a NE
                    aligned_labels.append(label_to_id["I-" + label[2:]]) #changing the rest to I-
                else:
                    aligned_labels.append(label_to_id[label]) #if label already an I- or O it stays the same
            previous_word_id = word_id #update the prev word for next iteration

        all_labels.append(aligned_labels) #appending list of labels to overall list

    tokenized_inputs["labels"] = all_labels #add the aligned labels to the tokenized input
    return tokenized_inputs

In [48]:
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True).remove_columns(["tokens", "ner_tags"])
tokenized_val = val_dataset.map(tokenize_and_align_labels, batched=True).remove_columns(["tokens", "ner_tags"])

Map:   0%|          | 0/30562 [00:00<?, ? examples/s]

Map:   0%|          | 0/3396 [00:00<?, ? examples/s]

In [49]:
dataset = Dataset.from_dict({"tokens": text_clean, "ner_tags": labels_clean})

In [50]:
len(dataset)

33958

In [55]:
from transformers import set_seed

# Set random seed!
SEED = 67
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list), id2label=id_to_label, label2id=label_to_id)
model_name = model_checkpoint.split("/")[-1]

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./roberta-ner",
    eval_strategy="epoch",       
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,  
    metric_for_best_model="eval_loss",
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,  
    processing_class=tokenizer,
    data_collator=data_collator
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
trainer.train()

c:\Users\M.Walavalkar\AppData\Local\anaconda3\envs\thesis_ner\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./finetuned_roberta")
tokenizer.save_pretrained("./finetuned_roberta")

# Testing

In [1]:
import torch
from transformers import AutoModelForTokenClassification, RobertaTokenizerFast

In [9]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328")
tokenizer = RobertaTokenizerFast.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328", add_prefix_space=True)
id_to_label = model.config.id2label

In [ ]:
def predict_sentence(tokens):
    """
    this returns a label for each input token in a sentence
    - claude troubleshooting used
    """
    #tokenizing the text
    encoding = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=512)
    word_ids = encoding.word_ids() #mapping each subword to its original word
    encoding_on_device = {key: val.to(model.device) for key, val in encoding.items()} #needed because tensor error otherwise

    with torch.no_grad():
        outputs = model(**encoding_on_device) #dont need to calculate gradients

    predicted_ids = torch.argmax(outputs.logits, dim=-1)[0].tolist() #argmax to get the label with the highest score
    predicted_subword_labels = [id_to_label[i] for i in predicted_ids] #convert numeric labels to text labels

    word_level_labels = [] #storing one label per original word
    previous_word_id = None

    for subword_index, word_id in enumerate(word_ids):
        if word_id is None: #skipping special tokens
            continue
        if word_id != previous_word_id: #checking if its a first subword of a word
            word_level_labels.append(predicted_subword_labels[subword_index]) #then add the pred for the first subword to the list
        previous_word_id = word_id #update so the next subword of the same word can be skipped

    if len(word_level_labels) < len(tokens): #adding padding to the list 
        word_level_labels += ["O"] * (len(tokens) - len(word_level_labels))

    return word_level_labels

def predict_and_save_conll(test_sents, test_labels, output_path):
    """
    this runs predictions on all sentences and then saves results to a conll file
    """
    true_labels = []
    predicted_labels = []

    with open(output_path, "w", encoding="utf-8") as infile:
        for tokens, gold_labels in zip(test_sents, test_labels):
            predicted = predict_sentence(tokens)
            true_labels.append(gold_labels)
            predicted_labels.append(predicted)

            for token, gold, pred in zip(tokens, gold_labels, predicted):
                infile.write(f"{token}\t{pred}\n")
            infile.write("\n")

    print(f"Predictions saved to: {output_path}")
    return true_labels, pred_labels

def evaluate(gold_sequences, pred_sequences):
    """
    evaluate gliner's predictions using seqeval strict span-level evaluation
    """
    report = classification_report(gold_sequences,pred_sequences, scheme=IOB2, mode="strict",output_dict=False,zero_division=0)
    print(report)

In [11]:
test_sents, test_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/my_data/final_dataset_1st_may.conll")

In [12]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328")
tokenizer = RobertaTokenizerFast.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328", add_prefix_space=True)

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328/output


In [20]:
##roberta -- existing legal data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_OS/model_output/checkpoint-5328-preds")
evaluate(test_labels, pred_labels)

              precision    recall  f1-score   support

       COURT       0.30      0.08      0.12       106
        DATE       0.47      0.07      0.12       132
         GPE       0.42      0.13      0.20       299
JURISDICTION       0.00      0.00      0.00       153
         LAW       0.00      0.00      0.00        65
         ORG       0.13      0.10      0.11       221
      PERSON       0.41      0.25      0.31       186
   PROVISION       0.16      0.11      0.14        87
 TAX_CONCEPT       0.00      0.00      0.00       353
    TAX_TYPE       0.00      0.00      0.00        86

   micro avg       0.24      0.08      0.12      1688
   macro avg       0.19      0.07      0.10      1688
weighted avg       0.20      0.08      0.11      1688



In [21]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_LLM/model_output/checkpoint-5344")
tokenizer = RobertaTokenizerFast.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_LLM/model_output/checkpoint-5344", add_prefix_space=True)

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_LLM/model_output/checkpoint-5344-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_LLM/model_output/checkpoint-5344-preds


In [22]:
##roberta -- llm labeled data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_LLM/model_output/checkpoint-5344-preds")
evaluate(test_labels, pred_labels)

              precision    recall  f1-score   support

       COURT       0.37      0.71      0.48       106
        DATE       0.74      0.73      0.74       132
         GPE       0.57      0.61      0.59       299
JURISDICTION       0.44      0.39      0.41       153
         LAW       0.15      0.42      0.22        65
         ORG       0.29      0.29      0.29       221
      PERSON       0.29      0.13      0.18       186
   PROVISION       0.10      0.11      0.11        87
 TAX_CONCEPT       0.18      0.06      0.09       353
    TAX_TYPE       0.39      0.36      0.37        86

   micro avg       0.38      0.35      0.36      1688
   macro avg       0.35      0.38      0.35      1688
weighted avg       0.36      0.35      0.34      1688



In [24]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_MIX/model_output/checkpoint-10101")
tokenizer = RobertaTokenizerFast.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_MIX/model_output/checkpoint-10101", add_prefix_space=True)

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_MIX/model_output/checkpoint-10101-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_MIX/model_output/checkpoint-10101-preds


In [25]:
##roberta -- mixed data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/ROBERTA_MIX/model_output/checkpoint-10101-preds")
evaluate(test_labels, pred_labels)

              precision    recall  f1-score   support

       COURT       0.36      0.65      0.46       106
        DATE       0.69      0.70      0.70       132
         GPE       0.64      0.54      0.59       299
JURISDICTION       0.49      0.41      0.45       153
         LAW       0.17      0.38      0.23        65
         ORG       0.28      0.27      0.28       221
      PERSON       0.31      0.30      0.30       186
   PROVISION       0.10      0.09      0.10        87
 TAX_CONCEPT       0.20      0.06      0.10       353
    TAX_TYPE       0.23      0.29      0.26        86

   micro avg       0.38      0.34      0.36      1688
   macro avg       0.35      0.37      0.35      1688
weighted avg       0.37      0.34      0.34      1688

